# Text analysis reports

`format_analysis()` presents analysis dictionaries and diagnostic tables as compact text. The underlying reports remain available for numerical work. Formatting does not fit or evaluate models.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from mnplib import (
    Miscoding, Inaccuracy, Surfeit, Nescience,
    NescienceClassifier, NescienceRegressor, TimeSeries, AnomalyDetector,
)
from mnplib.reporting import format_analysis

rng = np.random.default_rng(4)
X = pd.DataFrame(rng.integers(0, 2, size=(400, 3)),
                 columns=['signal', 'context', 'noise'])
y = 2 * X.signal.to_numpy() + X.context.to_numpy()
model = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)


## Metric analysis

Each metric exposes `model_analysis()`. Miscoding also provides feature tables and subset diagnostics.


In [ ]:
for metric_class in (Miscoding, Inaccuracy, Surfeit, Nescience):
    metric = metric_class().fit(X, y)
    report = metric.model_analysis(model)
    print(format_analysis(report))
    print()


In [ ]:
miscoding = Miscoding().fit(X, y)
selected_features = miscoding.select_features()
report = miscoding.subset_analysis(selected_features)
print(format_analysis(report))
print(format_analysis(miscoding.feature_analysis()))


## Automated classification and regression

`analysis()` describes the selected candidate. Its estimator score was measured on training data; use separate evaluation data to assess generalization.


In [ ]:
classifier = NescienceClassifier(models=['decision_tree'], random_state=0).fit(X, y)
print(format_analysis(classifier.analysis()))

target = 3 * X.signal - 2 * X.context + 0.1 * X.noise
regressor = NescienceRegressor(models=['linear_regression']).fit(X, target)
print(format_analysis(regressor.analysis()))


## Time series

The selected forecast model report labels its lagged training score and selected lags. Lag analysis is available as a separate table.


In [ ]:
series = np.tile([0., 1., 2., 1.], 60)
forecaster = TimeSeries(window_size=2, models=['autoregressive']).fit(series)
print(format_analysis(forecaster.analysis()))
print(format_analysis(forecaster.lag_analysis()))


## Anomalies

Anomaly reports summarize model-relative mismatches and the status of the correction-pattern analysis. Here, deliberate prediction errors provide several correction patterns.


In [ ]:
predictions = model.predict(X)
predictions[::3] = (predictions[::3] + 1) % 4
detector = AnomalyDetector(task='classification').fit(X, y, predictions=predictions)
print(format_analysis(detector.analysis()))


## Unreliable subset estimates

Sparse joint distributions are reported explicitly as unreliable, with NaN scores and a failure reason.


In [ ]:
sparse_X = np.arange(60).reshape(20, 3).astype(str)
sparse_y = np.arange(20).astype(str)
sparse_metric = Miscoding().fit(sparse_X, sparse_y)
print(format_analysis(sparse_metric.subset_analysis([0, 1])))
